In [3]:
import numpy as np
import cv2
import joblib
import tensorflow as tf
import numpy.linalg as npl

# Load models
scaler = joblib.load("models/scaler.pkl")
var_sel = joblib.load("models/var.pkl")
rfe = joblib.load("models/rfe.pkl")
mu = joblib.load("models/mu.pkl")
cov = joblib.load("models/cov.pkl")
knn = joblib.load("models/knn.pkl")
robust = joblib.load("models/robust.pkl")
lr = joblib.load("models/lr.pkl")
svm_fruit = joblib.load("models/svm_fruit.pkl")
# EfficientNet
model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", pooling="avg"
)
preprocess = tf.keras.applications.efficientnet.preprocess_input

print("All models loaded")

All models loaded


In [5]:
def extract_handcrafted(img):
    from skimage.feature import graycomatrix, graycoprops
    from scipy.stats import entropy

    img = cv2.resize(img, (224,224))
    img = img.astype(np.float32)

    # RGB (6)
    b,g,r = cv2.split(img)
    rgb = [r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()]

    # HSV (6)
    hsv = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2HSV)
    h,s,v = hsv[:,:,0].astype(np.float32), hsv[:,:,1], hsv[:,:,2]

    h_rad = h * (2*np.pi/180)
    cx, sx = np.cos(h_rad).mean(), np.sin(h_rad).mean()
    h_mean = np.arctan2(sx, cx)
    R = np.sqrt(cx**2 + sx**2)
    h_std = np.sqrt(-2*np.log(R + 1e-6))

    hsv_feat = [h_mean, s.mean(), v.mean(), h_std, s.std(), v.std()]

    # LAB (6)
    lab = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2LAB)
    l,a,b2 = lab[:,:,0], lab[:,:,1], lab[:,:,2]
    lab_feat = [l.mean(), l.std(), a.mean(), a.std(), b2.mean(), b2.std()]

    # TEXTURE (5)
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2GRAY)

    lap = cv2.Laplacian(gray, cv2.CV_64F).var()

    g = (gray // 8).astype(np.uint8)
    glcm = graycomatrix(g, distances=[1], angles=[0], levels=32, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')[0,0]
    energy = graycoprops(glcm, 'energy')[0,0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0,0]

    hist,_ = np.histogram(gray, bins=256, range=(0,255), density=True)
    ent = entropy(hist + 1e-6)

    texture = [lap, contrast, energy, homogeneity, ent]

    # SHAPE (6)
    _,th = cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours,_ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)
        peri = cv2.arcLength(c, True)

        circ = 4*np.pi*area/(peri*peri + 1e-6)

        hull = cv2.convexHull(c)
        hull_area = cv2.contourArea(hull)
        solidity = area/(hull_area + 1e-6)

        x,y,w,h = cv2.boundingRect(c)
        aspect = w/(h+1e-6)
        extent = area/(w*h + 1e-6)
    else:
        area=peri=circ=solidity=aspect=extent=0

    shape = [area, peri, circ, solidity, aspect, extent]

    # DARK PIXEL (1)
    dark_ratio = (gray < 50).mean()

    return np.array(rgb + hsv_feat + lab_feat + texture + shape + [dark_ratio], dtype=np.float32)

In [ ]:
# final prediction
def predict(img_path):
    
    # ===== Load Image =====
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Image not found")

    # ===== Handcrafted (32) =====
    hand = extract_handcrafted(img)

    # ===== EfficientNet (1280) =====
    img_resized = cv2.resize(img, (224,224))
    x = preprocess(np.expand_dims(img_resized,0))
    emb = model.predict(x, verbose=0).flatten()

    # ===== Combine (1312) =====
    feat = np.concatenate([emb, hand]).reshape(1,-1)

    # ===== Safety Check =====
    if feat.shape[1] != scaler.mean_.shape[0]:
        raise ValueError(f"Feature mismatch: {feat.shape[1]} vs trained {scaler.mean_.shape[0]}")

    # ===== Preprocessing =====
    x = scaler.transform(feat)
    x = var_sel.transform(x)
    x = rfe.transform(x) ## apply same feature selection mask

    # ===== Fruit Type =====
    fruit = svm_fruit.predict(x)[0]

    # ===== Mahalanobis =====
    mu_f = mu[fruit]
    inv = npl.inv(cov[fruit])

    diff = x[0] - mu_f
    M = np.sqrt(diff @ inv @ diff)

    # ===== KNN =====
    K = knn.kneighbors(x)[0].mean()

    # ===== Final Score =====
    MK = robust.transform([[M, K]])
    p = lr.predict_proba(MK)[0][1]

    freshness = 100 * ( 1- p)

    # ===== Label =====
    if freshness > 75:
        label = "Very Fresh"
    elif freshness > 50:
        label = "Fresh"
    elif freshness > 25:
        label = "Slightly Degraded"
    else:
        label = "Rotten"

    return {
        "fruit": fruit,
        "freshness_score": round(freshness,2),
        "label": label
    }

In [56]:
# prediction
result = predict("fresh_banana.png")

print("\n===== RESULT =====")
print("Fruit:", result["fruit"])
print("Freshness:", result["freshness_score"], "%")
print("Category:", result["label"])


===== RESULT =====
Fruit: Banana
Freshness: 0.73 %
Category: Rotten


# Diagnisis

In [39]:
import numpy as np
import joblib
import numpy.linalg as npl

# Load everything
X_train = np.load("artifacts/X_train.npy")
y_train = np.load("artifacts/y_train.npy")
ft_train = np.load("artifacts/ft_train.npy", allow_pickle=True)
mu_dict  = joblib.load("models/mu.pkl")
cov_dict = joblib.load("models/cov.pkl")
knn      = joblib.load("models/knn.pkl")
robust   = joblib.load("models/robust.pkl")
lr       = joblib.load("models/lr.pkl")

# Recompute M
def mahal_batch(X, ft):
    M = np.zeros(len(X))
    for f in np.unique(ft):
        idx = (ft == f)
        diff = X[idx] - mu_dict[f]
        inv  = npl.inv(cov_dict[f])
        M[idx] = np.sqrt(np.sum(diff @ inv * diff, axis=1))
    return M

M_train = mahal_batch(X_train, ft_train)
K_train = knn.kneighbors(X_train)[0].mean(axis=1)

# Diagnostics
print("=== M (Mahalanobis) ===")
print("  Fresh  mean:", M_train[y_train==0].mean())
print("  Rotten mean:", M_train[y_train==1].mean())

print("\n=== K (KNN distance) ===")
print("  Fresh  mean:", K_train[y_train==0].mean())
print("  Rotten mean:", K_train[y_train==1].mean())

print("\n=== Logistic Regression ===")
print("  classes_:", lr.classes_)
print("  coef_   :", lr.coef_)
print("  intercept:", lr.intercept_)

=== M (Mahalanobis) ===
  Fresh  mean: 10.648823332224897
  Rotten mean: 22.81843506039685

=== K (KNN distance) ===
  Fresh  mean: 4.6369033078616875
  Rotten mean: 10.09915599929105

=== Logistic Regression ===
  classes_: [0 1]
  coef_   : [[1.82741849 5.10818018]]
  intercept: [0.46785925]


In [60]:
def predict(img_path):
    
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Image not found")

    hand = extract_handcrafted(img)

    img_resized = cv2.resize(img, (224,224))
    x = preprocess(np.expand_dims(img_resized,0))
    emb = model.predict(x, verbose=0).flatten()

    feat = np.concatenate([emb, hand]).reshape(1,-1)

    if feat.shape[1] != scaler.mean_.shape[0]:
        raise ValueError(f"Feature mismatch: {feat.shape[1]} vs trained {scaler.mean_.shape[0]}")

    x = scaler.transform(feat)
    x = var_sel.transform(x)
    x = rfe.transform(x)

    fruit = svm_fruit.predict(x)[0]

    mu_f = mu[fruit]
    inv = npl.inv(cov[fruit])
    diff = x[0] - mu_f
    M = np.sqrt(diff @ inv @ diff)

    K = knn.kneighbors(x)[0].mean()

    # ===== DEBUG =====
    print(f"[DEBUG] Image     : {img_path}")
    print(f"[DEBUG] Fruit     : {fruit}")
    print(f"[DEBUG] M (raw)   : {M:.4f}   (expected: fresh~10.6, rotten~22.8)")
    print(f"[DEBUG] K (raw)   : {K:.4f}   (expected: fresh~4.6,  rotten~10.1)")
    MK_scaled = robust.transform([[M, K]])
    print(f"[DEBUG] M_scaled  : {MK_scaled[0][0]:.4f}")
    print(f"[DEBUG] K_scaled  : {MK_scaled[0][1]:.4f}")
    p_rotten = lr.predict_proba(MK_scaled)[0][1]
    print(f"[DEBUG] P(rotten) : {p_rotten:.4f}")
    print(f"[DEBUG] freshness : {100*(1-p_rotten):.2f}%")
    print("-" * 45)
    print("[DEBUG] x mean:", x[0].mean())
    print("[DEBUG] x std :", x[0].std())
    print("[DEBUG] x min :", x[0].min())
    print("[DEBUG] x max :", x[0].max())
    # ===== END DEBUG =====

    MK = robust.transform([[M, K]])
    p = lr.predict_proba(MK)[0][1]
    freshness = 100 * (1 - p)

    if freshness > 75:
        label = "Very Fresh"
    elif freshness > 50:
        label = "Fresh"
    elif freshness > 25:
        label = "Slightly Degraded"
    else:
        label = "Rotten"

    return {
        "fruit": fruit,
        "freshness_score": round(freshness, 2),
        "label": label
    }

In [61]:
# prediction
result = predict("fresh_banana_green.jpg")

print("\n===== RESULT =====")
print("Fruit:", result["fruit"])
print("Freshness:", result["freshness_score"], "%")
print("Category:", result["label"])

[DEBUG] Image     : fresh_banana_green.jpg
[DEBUG] Fruit     : Banana
[DEBUG] M (raw)   : 15.5318   (expected: fresh~10.6, rotten~22.8)
[DEBUG] K (raw)   : 8.8633   (expected: fresh~4.6,  rotten~10.1)
[DEBUG] M_scaled  : 0.1005
[DEBUG] K_scaled  : 0.2605
[DEBUG] P(rotten) : 0.8789
[DEBUG] freshness : 12.11%
---------------------------------------------
[DEBUG] x mean: -0.46732795
[DEBUG] x std : 1.0120951
[DEBUG] x min : -2.2441604
[DEBUG] x max : 5.053281

===== RESULT =====
Fruit: Banana
Freshness: 12.11 %
Category: Rotten


In [46]:
# prediction
result = predict("fresh_capsicum.jpg")

print("\n===== RESULT =====")
print("Fruit:", result["fruit"])
print("Freshness:", result["freshness_score"], "%")
print("Category:", result["label"])

[DEBUG] Image     : fresh_capsicum.jpg
[DEBUG] Fruit     : Capsicum
[DEBUG] M (raw)   : 28.1537   (expected: fresh~10.6, rotten~22.8)
[DEBUG] K (raw)   : 9.7986   (expected: fresh~4.6,  rotten~10.1)
[DEBUG] M_scaled  : 1.2964
[DEBUG] K_scaled  : 0.4265
[DEBUG] P(rotten) : 0.9934
[DEBUG] freshness : 0.66%
---------------------------------------------
[DEBUG] x mean: -0.2366248
[DEBUG] x std : 1.0004973
[DEBUG] x min : -1.656506
[DEBUG] x max : 4.8262887

===== RESULT =====
Fruit: Capsicum
Freshness: 0.66 %
Category: Rotten


In [43]:
import joblib
import numpy as np

X_train = np.load("artifacts/X_train.npy")
y_train = np.load("artifacts/y_train.npy")
ft_train = np.load("artifacts/ft_train.npy", allow_pickle=True)
mu = joblib.load("models/mu.pkl")

# Compare stored mu vs actual fresh/rotten means in feature space
for fruit in ["Apple", "Capsicum"]:
    idx_fresh  = (ft_train == fruit) & (y_train == 0)
    idx_rotten = (ft_train == fruit) & (y_train == 1)
    
    dist_fresh  = np.linalg.norm(mu[fruit] - X_train[idx_fresh].mean(axis=0))
    dist_rotten = np.linalg.norm(mu[fruit] - X_train[idx_rotten].mean(axis=0))
    
    print(f"{fruit}: mu closer to FRESH={dist_fresh:.3f}  ROTTEN={dist_rotten:.3f}")

Apple: mu closer to FRESH=0.000  ROTTEN=4.004
Capsicum: mu closer to FRESH=0.000  ROTTEN=5.959


In [48]:
import os, random
from pathlib import Path

fresh_imgs  = [str(p) for p in Path("DataSet/Fresh/FreshApple").rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png"}][:3]
rotten_imgs = [str(p) for p in Path("DataSet/Rotten/RottenApple").rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png"}][:3]

print("=== FRESH APPLES ===")
for p in fresh_imgs:
    r = predict(p)
    print(f"{os.path.basename(p)} → {r['label']} ({r['freshness_score']}%)")

print("\n=== ROTTEN APPLES ===")
for p in rotten_imgs:
    r = predict(p)
    print(f"{os.path.basename(p)} → {r['label']} ({r['freshness_score']}%)")

=== FRESH APPLES ===
[DEBUG] Image     : DataSet\Fresh\FreshApple\a_f004.png
[DEBUG] Fruit     : Apple
[DEBUG] M (raw)   : 9.0000   (expected: fresh~10.6, rotten~22.8)
[DEBUG] K (raw)   : 3.3257   (expected: fresh~4.6,  rotten~10.1)
[DEBUG] M_scaled  : -0.5184
[DEBUG] K_scaled  : -0.7221
[DEBUG] P(rotten) : 0.0152
[DEBUG] freshness : 98.48%
---------------------------------------------
[DEBUG] x mean: -0.021911262
[DEBUG] x std : 0.88739747
[DEBUG] x min : -1.5426879
[DEBUG] x max : 3.7377706
a_f004.png → Very Fresh (98.48%)
[DEBUG] Image     : DataSet\Fresh\FreshApple\a_f005.png
[DEBUG] Fruit     : Apple
[DEBUG] M (raw)   : 8.9265   (expected: fresh~10.6, rotten~22.8)
[DEBUG] K (raw)   : 3.2915   (expected: fresh~4.6,  rotten~10.1)
[DEBUG] M_scaled  : -0.5254
[DEBUG] K_scaled  : -0.7281
[DEBUG] P(rotten) : 0.0146
[DEBUG] freshness : 98.54%
---------------------------------------------
[DEBUG] x mean: -0.2016401
[DEBUG] x std : 0.7424972
[DEBUG] x min : -1.1868235
[DEBUG] x max : 3.055

# new

In [4]:
# At top with other model loads
dasfs = joblib.load("models/dasfs.pkl")

In [6]:
def predict(img_path):

    # ===== Load Image =====
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Image not found")

    # ===== Handcrafted (32) =====
    hand = extract_handcrafted(img)

    # ===== EfficientNet (1280) =====
    img_resized = cv2.resize(img, (224, 224))
    x = preprocess(np.expand_dims(img_resized, 0))
    emb = model.predict(x, verbose=0).flatten()

    # ===== Combine (1312) =====
    feat = np.concatenate([emb, hand]).reshape(1, -1)

    # ===== Safety Check =====
    if feat.shape[1] != scaler.mean_.shape[0]:
        raise ValueError(f"Feature mismatch: {feat.shape[1]} vs trained {scaler.mean_.shape[0]}")

    # ===== Preprocessing =====
    x = scaler.transform(feat)
    x = var_sel.transform(x)
    x = rfe.transform(x)

    # ===== Fruit Type =====
    fruit = svm_fruit.predict(x)[0]

    # ===== Mahalanobis =====
    mu_f = mu[fruit]
    inv  = npl.pinv(cov[fruit])
    diff = x[0] - mu_f
    M    = np.sqrt(diff @ inv @ diff)

    # ===== KNN =====
    K = knn.kneighbors(x)[0].mean()

    # ===== DASFS Score =====
    if fruit in dasfs:
        d        = dasfs[fruit]
        proj     = x[0] @ d["axis"]
        p_fresh  = d["p_fresh"]
        p_rotten = d["p_rotten"]
        spread   = (d["spread_fresh"] + d["spread_rotten"]) / 2

        dasfs_score = float(np.clip(
            (p_rotten - proj) / (p_rotten - p_fresh + 1e-8), 0, 1
        ))
        midpoint   = (p_fresh + p_rotten) / 2
        confidence = float(np.clip(
            abs(proj - midpoint) / (spread + 1e-8), 0, 1
        ))
    else:
        # Fallback to original method if fruit not in dasfs
        MK          = robust.transform([[M, K]])
        p           = lr.predict_proba(MK)[0][1]
        dasfs_score = 1 - p
        confidence  = 0.5

    # ===== Hybrid Fusion (DASFS + Original LR) =====
    MK       = robust.transform([[M, K]])
    p_lr     = lr.predict_proba(MK)[0][1]
    lr_score = 1 - p_lr

    # Weight by confidence — trust DASFS more when confident
    w         = confidence
    freshness = 100 * (w * dasfs_score + (1 - w) * lr_score)

    # ===== Label =====
    if freshness > 75:
        label = "Very Fresh"
    elif freshness > 50:
        label = "Mostly Fresh"
    elif freshness > 25:
        label = "Slightly Degraded"
    else:
        label = "Rotten"

    return {
        "fruit"          : fruit,
        "freshness_score": round(freshness, 2),
        "label"          : label,
        "confidence"     : round(confidence * 100, 1),
        "dasfs_score"    : round(dasfs_score * 100, 2),
        "lr_score"       : round(lr_score * 100, 2)
    }

In [64]:
result = predict("fresh_banana.png")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Banana
Freshness Score: 100.0 %
Category       : Very Fresh
Confidence     : 100.0 %
DASFS Score    : 100.0 %
LR Score       : 0.73 %


In [65]:
result = predict("fresh_banana_green.jpg")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Banana
Freshness Score: 100.0 %
Category       : Very Fresh
Confidence     : 100.0 %
DASFS Score    : 100.0 %
LR Score       : 12.11 %


In [66]:
result = predict("rotten_banana.webp")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Banana
Freshness Score: 16.12 %
Category       : Rotten
Confidence     : 100.0 %
DASFS Score    : 16.12 %
LR Score       : 2.03 %


In [68]:
result = predict("rotten_banana_more.jpg")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Banana
Freshness Score: 0.0 %
Category       : Rotten
Confidence     : 100.0 %
DASFS Score    : 0.0 %
LR Score       : 0.0 %


In [69]:
result = predict("fresh_potato.jpg")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Potato
Freshness Score: 100.0 %
Category       : Very Fresh
Confidence     : 100.0 %
DASFS Score    : 100.0 %
LR Score       : 35.05 %


In [8]:
result = predict("rotten_apple.webp")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Apple
Freshness Score: 64.32 %
Category       : Mostly Fresh
Confidence     : 30.6 %
DASFS Score    : 37.21 %
LR Score       : 76.29 %


In [9]:
result = predict("rotten_apple_much.webp")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Apple
Freshness Score: 38.04 %
Category       : Slightly Degraded
Confidence     : 13.3 %
DASFS Score    : 55.57 %
LR Score       : 35.34 %


In [7]:
result = predict("fresh_capsicum.jpg")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Capsicum
Freshness Score: 71.5 %
Category       : Mostly Fresh
Confidence     : 93.0 %
DASFS Score    : 76.8 %
LR Score       : 0.66 %


In [10]:
result = predict("rotten_potato.jpg")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Potato
Freshness Score: 24.22 %
Category       : Rotten
Confidence     : 2.7 %
DASFS Score    : 51.27 %
LR Score       : 23.46 %


In [21]:
result = predict("test_img/rotten_apple_much.webp")

print("\n===== RESULT =====")
print("Fruit          :", result["fruit"])
print("Freshness Score:", result["freshness_score"], "%")
print("Category       :", result["label"])
print("Confidence     :", result["confidence"], "%")
print("DASFS Score    :", result["dasfs_score"], "%")
print("LR Score       :", result["lr_score"], "%")


===== RESULT =====
Fruit          : Apple
Freshness Score: 38.04 %
Category       : Slightly Degraded
Confidence     : 13.3 %
DASFS Score    : 55.57 %
LR Score       : 35.34 %
